# Phase 7 — Performance

## Pourquoi cette phase est importante

Les phases 5 et 6 ont déjà publié un rendement, une volatilité, un ratio de Sharpe et un drawdown maximal. Cette phase les reprend et y ajoute le ratio de Sortino ainsi que l'écart de suivi par rapport au fonds iShares.

Le ratio de Sharpe pénalise toute variation, à la hausse comme à la baisse. Le ratio de Sortino ne pénalise que les mois où le portefeuille ne fait pas mieux que le SARON.

L'écart de suivi mesure, chaque mois, l'écart de rendement entre le portefeuille et le fonds, une fois le rendement de la valeur liquidative converti en francs.

## Où en est le calcul

Le module est `src/performance/build_performance.py`. La commande est `python -m src.performance.build_performance`. Il écrit `data/06_performance/performance_metrics.csv`, une ligne par stratégie.

Il relit les rendements mensuels de la phase 5. Il retrouve le rendement, la volatilité, le ratio de Sharpe et le drawdown maximal déjà publiés dans `portfolio_metrics.csv`. Si un de ces quatre nombres diverge, le module s'arrête. Il retrouve aussi la performance cumulée et le drawdown de `portfolio_path.csv`. Le ratio de Sortino, l'écart de suivi et le ratio d'information sont calculés ici.

Les trois livrables sont le rapport `reports/performance/07_rapport_performance.ipynb`, l'analyse `notebooks/07_analyse_performance.ipynb`, et le tableau de bord `reports/performance/dashboard.html`. Le détail s'affiche au survol. La vérification est `tests/performance/phase_07_validation.ipynb`.


## Sources lues

Cette phase ne retourne pas aux cours bruts. Le rendement mensuel officiel est déjà dans la sortie de la phase 5. Revenir aux cours Yahoo créerait une deuxième série.

| Fichier | Origine | Colonnes | Usage |
|---|---|---|---|
| `data/04_portfolios/portfolio_monthly_returns.csv` | phase 5 | `month`, `strategy`, `portfolio_return`, `risk_free_return`, `excess_return` | Toutes les mesures de cette phase |
| `data/04_portfolios/portfolio_metrics.csv` | phase 5 | `annualized_return`, `annualized_volatility`, `sharpe_ratio`, `maximum_drawdown` | Contrôle : les quatre mesures déjà publiées doivent être retrouvées |
| `data/05_monitoring/portfolio_path.csv` | phase 6 | `cumulative_wealth`, `drawdown` | Contrôle : la performance cumulée et le drawdown |

La référence de l'écart de suivi est le fonds iShares dont la feuille `Performance` du même classeur publie le rendement mensuel total de la valeur liquidative. La devise est celle de la feuille `Holdings` : la colonne `Currency`, confirmée par les lignes dont le `FX Rate` vaut 1. Le passage en francs utilise le cours BNS de fin de mois de cette devise, `DS-004`. Un mois sans rendement publié, ou sans cours du mois et du mois précédent, arrête le calcul. Rien n'est interpolé.

Remontée vers le registre, si l'on veut la source des rendements de portefeuille : `DS-003` et `DS-004` pour les cours en CHF, `DS-005` pour le SARON, `DS-002` pour la liste des titres. Ces jeux ont déjà été consommés par la phase 5. La phase 7 ne rouvre pas les cours Yahoo. Elle rouvre le classeur du benchmark pour la feuille de performance, et le fichier de change pour le cours de la devise du fonds.

| Jeu non lu | Pourquoi |
|---|---|
| `DS-006` évaluation ODD | Le statut est Indeterminate. Il ne modifie pas un rendement. |


## Rendement, volatilité, Sharpe, drawdown maximal

Ces quatre formules sont celles de la phase 5. Elles sont répétées pour que cette page se lise seule. $R_{p,t}$ est `portfolio_return`. $E_{p,t}$ est `excess_return`. $T$ est le nombre de mois du fichier.

$$
R_p^{\mathrm{ann}} = \left(\prod_{t=1}^{T} (1 + R_{p,t})\right)^{12/T} - 1
$$

$$
\sigma_p^{\mathrm{ann}} = s(R_p) \times \sqrt{12}
$$

$s$ est l'écart-type d'échantillon, diviseur $T - 1$.

$$
\mathrm{Sharpe}_p = \frac{\bar E_p}{s(E_p)} \times \sqrt{12}
$$

La performance d'un placement de 1 CHF et le drawdown sont ceux de la phase 6 :

$$
W_t = \prod_{u \le t} (1 + R_{p,u})
\qquad
D_t = \frac{W_t}{\max_{u \le t} W_u} - 1
$$

Le drawdown maximal publié est le minimum de $D_t$. Il est négatif.

Un écart entre ces quatre nombres et `portfolio_metrics.csv` arrête le calcul. Un seul ratio de Sharpe est publié.


## Sortino

Le ratio de Sharpe traite un mois très favorable comme un mois très défavorable : les deux éloignent le résultat de la moyenne. Le ratio de Sortino ne retient au dénominateur que les mois d'excédent négatif, c'est-à-dire les mois qui ne font pas mieux que le SARON.

Pour un mois d'excédent positif, la contribution au dénominateur est zéro. Pour un mois négatif, c'est le carré de l'excédent. Le diviseur reste $T - 1$, le même que pour l'écart-type du Sharpe. On ne divise pas par le seul nombre de mois négatifs : ce choix rendrait deux portefeuilles incomparables.

$$
\sigma^- = \sqrt{ \frac{1}{T - 1} \sum_{t=1}^{T} \min(E_{p,t},\, 0)^2 }
$$

$$
\mathrm{Sortino}_p = \frac{\bar E_p}{\sigma^-} \times \sqrt{12}
$$

Si aucun mois n'est en dessous du SARON, $\sigma^-$ vaut zéro et le ratio de Sortino n'est pas publié. Aucun dénominateur de remplacement n'est introduit.


## Écart de suivi

La référence est le rendement mensuel total de la valeur liquidative publié par iShares, passé en francs. Ce n'est pas un rendement reconstruit avec les poids d'aujourd'hui.

La feuille `Performance` donne le chiffre $r_t$. La feuille `Historical` sert seulement à lire l'unité : sur un mois sans distribution, $r_t$ doit retrouver la variation de la valeur liquidative. S'il la retrouve une fois divisé par 100, le chiffre est en points de pourcentage. La feuille `Distributions` dit quels mois ont un détachement, et ces mois ne servent pas à cette vérification. La feuille `Holdings` dit la devise : une seule valeur dans `Currency`, et les lignes à `FX Rate` = 1 vérifient que le prix fois la quantité égale la valeur de marché.

$S_t$ est le nombre de francs pour une unité de cette devise, cours BNS de fin de mois. Le rendement en francs est

$$
R_{b,t} = (1 + R_{\mathrm{nav},t}) \times \frac{S_t}{S_{t-1}} - 1
$$

L'écart du mois $t$ est

$$
\Delta_t = R_{p,t} - R_{b,t}
$$

L'écart de suivi annualisé est l'écart-type d'échantillon de $\Delta_t$ :

$$
\mathrm{TE}_p = s(\Delta) \times \sqrt{12}
$$

Le ratio d'information rapporte l'écart moyen à cette distance.

$$
\mathrm{IR}_p = \frac{\bar\Delta}{s(\Delta)} \times \sqrt{12}
$$

Un ratio positif veut dire que la stratégie a, en moyenne sur la période, rendu plus que le fonds. Chaque poids avait été fixé avant le mois qu'il concerne.


## Fichier publié

`data/06_performance/performance_metrics.csv` contient une ligne par stratégie.

| Colonne | Sens |
|---|---|
| `strategy` | `equal_weight`, `minimum_volatility`, `maximum_sharpe` |
| `months`, `window_start`, `window_end` | La période commune, la même qu'en phase 5 |
| `annualized_return` | Même définition que la phase 5 |
| `annualized_volatility` | Même définition que la phase 5 |
| `sharpe_ratio` | Même définition que la phase 5 |
| `sortino_ratio` | Formule de cette page. Cellule vide si aucun mois n'est sous le SARON |
| `maximum_drawdown` | Drawdown maximal, minimum de $D_t$ |
| `tracking_error` | Contre le rendement en francs du fonds |
| `information_ratio` | Contre le même fonds. Cellule vide si l'écart de suivi est zéro |
| `months_below_risk_free` | Nombre de mois où l'excédent est strictement négatif |
| `reference` | Nom du fonds publié dans le classeur |

`data/06_performance/benchmark_monthly_return.csv` contient une ligne par mois : le rendement de la valeur liquidative dans la devise du fonds, les deux cours BNS, et le rendement en francs.

La cellule suivante lit ce fichier. Elle ne recalcule pas les poids.


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "src").is_dir():
        PROJECT_ROOT = candidate
        break
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))
import pandas as pd

published = PROJECT_ROOT / "data/06_performance/performance_metrics.csv"
print("présent" if published.exists() else "absent", published.relative_to(PROJECT_ROOT))
if published.exists():
    metrics = pd.read_csv(published)
    print(metrics.to_string(index=False))


présent data\06_performance\performance_metrics.csv
          strategy  months window_start window_end  annualized_return  annualized_volatility  sharpe_ratio  sortino_ratio  maximum_drawdown  tracking_error  information_ratio  months_below_risk_free                                             reference
      equal_weight     120      2016-10    2026-09           0.083651               0.124179      0.719068       1.142029         -0.170011        0.058872           0.334157                      47 iShares MSCI Global Sustainable Development Goals ETF
minimum_volatility     120      2016-10    2026-09           0.067909               0.110882      0.657889       1.033231         -0.192280        0.072472           0.046816                      45 iShares MSCI Global Sustainable Development Goals ETF
    maximum_sharpe     120      2016-10    2026-09           0.106017               0.128500      0.859457       1.382486         -0.202370        0.079060           0.516364               

# Dictionnaire des termes de cette note

Ce dictionnaire reprend les termes employés dans cette note. Il n'en ajoute pas. Les formules restent dans la section qui les écrit.

**Banque nationale suisse (BNS).** Source du cours de fin de mois de la devise du fonds.

**Drawdown.** Baisse de la performance cumulée depuis le plus haut déjà atteint. Le drawdown maximal publié est le minimum de cette série. Il est négatif.

**Écart de suivi.** Écart-type des écarts mensuels entre le rendement du portefeuille et le rendement en francs du fonds.

**Excédent.** Rendement du portefeuille moins le taux sans risque du mois. Il est déjà dans `excess_return`.

**Indeterminate.** Statut ODD du jeu `DS-006`. Il ne modifie pas un rendement.

**iShares.** Éditeur du proxy investissable, le fonds de comparaison. Le rendement utilisé est celui que sa feuille `Performance` publie pour la valeur liquidative. La référence stratégique est l'indice MSCI ACWI Sustainable Development Index (Net) : elle n'entre pas dans ce calcul.

**Performance cumulée.** Valeur d'un placement de 1 CHF.

**Ratio d'information.** Écart moyen avec le fonds, divisé par l'écart de suivi.

**Ratio de Sharpe.** Moyenne de l'excédent divisée par l'écart-type de l'excédent, puis annualisée. Il pénalise la hausse comme la baisse.

**Ratio de Sortino.** Même numérateur que le Sharpe. Le dénominateur ne retient que les mois d'excédent négatif. Si aucun mois n'est sous le SARON, le ratio n'est pas publié.

**`annualized_return`.** Résultat de la série mensuelle ramené à une année. Même définition que la phase 5.

**SARON.** Taux sans risque déjà incorporé dans `risk_free_return`.

**Valeur liquidative.** Valeur de la part du fonds publiée par iShares.

**Volatilité.** Écart-type d'échantillon des rendements du portefeuille, annualisé avec la racine de 12.

**Yahoo.** Source d'origine des cours. Cette phase ne les rouvre pas.
